# Export every Excel sheet to TSV + cleaned xlsx

**Data**: upload `main.xlsx` when the first cell asks.

Run the cells in order. <sub>Notebook generated from `src/convert.py`</sub>

> Generated files appear in the Colab Files pane (left sidebar).

In [ ]:
from pathlib import Path

files_needed = ["main.xlsx"]
missing = [f for f in files_needed if not Path(f).exists()]
if missing:
    try:
        from google.colab import files
        print("Upload:", ", ".join(missing))
        files.upload()   # select the file(s) in the picker
    except ImportError:
        print("Not running in Colab; place the dataset next to this notebook.")

In [ ]:
from pathlib import Path
import pandas as pd
file_path = Path("main.xlsx")
headerless = {"Sheet3"}

In [ ]:
data = {}
for sheet in ["Sheet1", "midterm result", "Final", "Sheet3"]:
    if sheet in headerless:
        data[sheet] = pd.read_excel(file_path, sheet_name=sheet, header=None)
    else:
        df = pd.read_excel(file_path, sheet_name=sheet)
        df = df.dropna(axis=1, how="all")
        df.columns = [str(c).strip() for c in df.columns]
        data[sheet] = df

for sheet, df in data.items():
    tsv_path = Path(f"main_{sheet.replace(' ', '_').lower()}.tsv")
    df.to_csv(tsv_path, sep="\t", index=False, header=sheet not in headerless)
    print(f"{tsv_path.name}  {df.shape}")

with pd.ExcelWriter("main_clean.xlsx", engine="openpyxl") as writer:
    for sheet, df in data.items():
        df.to_excel(writer, sheet_name=sheet, index=False, header=sheet not in headerless)
    print("main_clean.xlsx", list(data))